<a href="https://colab.research.google.com/github/riskoptima/blog/blob/master/The_most_sophisticated_RNSKCalculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.stats import norm
from scipy.optimize import brentq

class RNSKCalculator:
    """Calculates Risk-Neutral Skewness for contango prediction"""

    def __init__(self, client):
        self.client = client
        # Common futures root -> options parent candidates on GLBX
        self.root_to_option_parents = {
            'ZW': ['OZW.OPT', 'ZW.OPT'],
            'ZC': ['OZC.OPT', 'ZC.OPT', 'C.OPT'],
            'ZS': ['OZS.OPT', 'ZS.OPT'],
            'GC': ['OGC.OPT', 'OG.OPT', 'GC.OPT'],
            'SI': ['SO.OPT', 'OSI.OPT', 'SI.OPT'],
            'HG': ['HXE.OPT', 'HX.OPT', 'HC.OPT', 'OHG.OPT', 'HG.OPT'],
            'CL': ['LO.OPT', 'OCL.OPT', 'CL.OPT'],
            'NG': ['ON.OPT', 'ONG.OPT', 'NG.OPT'],
            'ES': ['EW.OPT', 'OES.OPT', 'ES.OPT'],
            'NQ': ['NQ.OPT', 'ONQ.OPT', 'EW3.OPT'],
        }
        self.primary_dataset = "GLBX.MDP3"

    def _get_option_parent_candidates(self, underlying: str):
        """Infer likely option parent symbols from the futures underlying"""
        try:
            root = underlying[:2].upper()
            candidates = self.root_to_option_parents.get(root, [])
            return candidates if candidates else []
        except Exception:
            return []

    def fetch_options_data(self, underlying: str, date: str):
        """Fetch options definitions and quotes for RNSK calculation"""
        try:
            date_dt = datetime.strptime(date, "%Y-%m-%d")
            end_dt = date_dt + timedelta(days=1)
            start_iso = date_dt.isoformat()
            end_iso = end_dt.isoformat()

            # Get futures price
            futures_data = self.client.timeseries.get_range(
                dataset="GLBX.MDP3",
                symbols=[underlying],
                schema="ohlcv-1d",
                start=start_iso,
                end=end_iso
            )
            futures_df = futures_data.to_df().reset_index()
            S = futures_df['close'].iloc[0] / 100.0

            # Get options definitions
            option_parents = self._get_option_parent_candidates(underlying)
            definitions_frames = []
            tried_parents = []

            for parent in option_parents:
                try:
                    definitions_data = self.client.timeseries.get_range(
                        dataset=self.primary_dataset,
                        symbols=[parent],
                        schema="definition",
                        stype_in="parent",
                        start=start_iso,
                        end=end_iso
                    )
                    df_parent = definitions_data.to_df().reset_index()
                    if not df_parent.empty:
                        definitions_frames.append(df_parent)
                        break
                except Exception:
                    pass
                finally:
                    tried_parents.append(parent)

            if len(definitions_frames) == 0:
                # Autodiscover via ALL_SYMBOLS
                try:
                    defs_all = self.client.timeseries.get_range(
                        dataset=self.primary_dataset,
                        symbols=["ALL_SYMBOLS"],
                        schema="definition",
                        stype_in="parent",
                        start=start_iso,
                        end=end_iso
                    ).to_df().reset_index()
                    if not defs_all.empty:
                        defs_under = defs_all[defs_all['underlying'] == underlying]
                        if defs_under.empty:
                            root = underlying[:2]
                            defs_under = defs_all[defs_all.get('underlying', '').astype(str).str.startswith(root)]
                        if not defs_under.empty:
                            definitions_frames.append(defs_under)
                            tried_parents.append('ALL_SYMBOLS_DISCOVERY')
                except Exception:
                    pass
                if len(definitions_frames) == 0:
                    raise ValueError(f"No options definitions for {underlying}. Tried: {', '.join(tried_parents)}")

            definitions_df = pd.concat(definitions_frames, ignore_index=True)

            # Filter options
            idx_root = underlying[:2].upper()
            relaxed_used = False
            if idx_root in ("ES", "NQ"):
                relaxed_used = True
            else:
                exact_defs = definitions_df[definitions_df['underlying'] == underlying]
                if exact_defs.empty:
                    root = underlying[:2]
                    relaxed_defs = definitions_df[definitions_df.get('underlying', '').astype(str).str.startswith(root)]
                    if relaxed_defs.empty:
                        raise ValueError(f"No options for {underlying} after filtering. Tried: {', '.join(tried_parents)}")
                    definitions_df = relaxed_defs
                    relaxed_used = True
                else:
                    definitions_df = exact_defs

            # Normalize and filter
            definitions_df['strike'] = definitions_df['strike_price'] / 100.0
            definitions_df['expiration_dt'] = pd.to_datetime(definitions_df['expiration'], utc=True, errors='coerce')
            calc_date = pd.to_datetime(date, utc=True)
            definitions_df = definitions_df.dropna(subset=['expiration_dt'])
            definitions_df['days_to_exp'] = (definitions_df['expiration_dt'] - calc_date).dt.days

            future_defs = definitions_df[definitions_df['days_to_exp'] >= 1]
            if future_defs.empty:
                future_defs = definitions_df.copy()
            nearest_exp = future_defs.sort_values('days_to_exp').iloc[0]['expiration_dt']
            defs_nearest = definitions_df[definitions_df['expiration_dt'] == nearest_exp]

            # Strike/time windows
            root_for_opts = underlying[:2].upper()
            moneyness_lower = 0.8
            moneyness_upper = 1.2
            quote_minutes = 30
            max_strikes = 500

            if root_for_opts in ("ES", "NQ"):
                moneyness_lower = 0.7
                moneyness_upper = 1.3
                quote_minutes = 90
                max_strikes = 1000
            elif root_for_opts in ("ZC", "ZW", "ZS"):
                moneyness_lower = 0.7
                moneyness_upper = 1.3
                quote_minutes = 120
                max_strikes = 800

            strike_min = max(1e-9, S * moneyness_lower)
            strike_max = S * moneyness_upper
            defs_nearest = defs_nearest[(defs_nearest['strike'] >= strike_min) & (defs_nearest['strike'] <= strike_max)]

            if len(defs_nearest) > max_strikes:
                defs_nearest['abs_moneyness'] = (defs_nearest['strike'] / S - 1.0).abs()
                defs_nearest = defs_nearest.sort_values('abs_moneyness').head(max_strikes)

            defs_nearest['option_type'] = defs_nearest.apply(self._infer_option_type, axis=1)
            defs_nearest = defs_nearest.dropna(subset=['option_type'])

            expiration_date = nearest_exp
            T = max(1, (expiration_date - calc_date).days) / 365.0

            # Get quotes
            symbols_to_fetch = defs_nearest['symbol'].tolist()
            chunk_size = 400
            quotes_frames = []
            attempt_windows = [quote_minutes]

            if root_for_opts in ("ES", "NQ"):
                attempt_windows = [quote_minutes, 240, 1440]
            elif root_for_opts in ("ZC", "ZW", "ZS"):
                attempt_windows = [quote_minutes, 480, 1440]
            else:
                attempt_windows = [quote_minutes, 120, 480, 1440]

            attempt_dates = [date_dt]
            if root_for_opts in ("ES", "NQ", "ZC", "ZW", "ZS"):
                attempt_dates = [date_dt - timedelta(days=i) for i in range(0, 5)]
            else:
                attempt_dates = [date_dt - timedelta(days=i) for i in range(0, 3)]

            found_any = False
            for dt_try in attempt_dates:
                for win in attempt_windows:
                    quotes_frames = []
                    start_try = (dt_try + timedelta(hours=0)).isoformat()
                    end_try = (dt_try + timedelta(days=1)).isoformat()
                    start_window = (dt_try + timedelta(days=1) - timedelta(minutes=win)).isoformat()
                    start_use = start_window if win < 1440 else start_try

                    for i in range(0, len(symbols_to_fetch), chunk_size):
                        chunk = symbols_to_fetch[i:i+chunk_size]
                        try:
                            quotes_data = self.client.timeseries.get_range(
                                dataset=self.primary_dataset,
                                symbols=chunk,
                                schema="mbp-1",
                                start=start_use,
                                end=end_try
                            )
                            qdf = quotes_data.to_df().reset_index()
                            if not qdf.empty:
                                quotes_frames.append(qdf)
                        except Exception:
                            pass

                    if len(quotes_frames) > 0:
                        found_any = True
                        break
                if found_any:
                    break

            if not found_any:
                # Fallback: trades schema
                trades_frames = []
                for dt_try in attempt_dates:
                    for win in attempt_windows:
                        trades_frames = []
                        start_try = (dt_try + timedelta(hours=0)).isoformat()
                        end_try = (dt_try + timedelta(days=1)).isoformat()
                        start_window = (dt_try + timedelta(days=1) - timedelta(minutes=win)).isoformat()
                        start_use = start_window if win < 1440 else start_try

                        for i in range(0, len(symbols_to_fetch), chunk_size):
                            chunk = symbols_to_fetch[i:i+chunk_size]
                            try:
                                trades_data = self.client.timeseries.get_range(
                                    dataset=self.primary_dataset,
                                    symbols=chunk,
                                    schema="trades",
                                    start=start_use,
                                    end=end_try
                                )
                                tdf = trades_data.to_df().reset_index()
                                if not tdf.empty:
                                    trades_frames.append(tdf)
                            except Exception:
                                pass

                        if len(trades_frames) > 0:
                            break
                    if len(trades_frames) > 0:
                        break

                if len(trades_frames) == 0:
                    raise ValueError("No quotes returned for filtered options set")

                quotes_df = pd.concat(trades_frames, ignore_index=True)
                latest_quotes = quotes_df.groupby('symbol').last()
                if 'price' in latest_quotes.columns:
                    latest_quotes['premium'] = latest_quotes['price'] / 100.0
                else:
                    raise ValueError("Trades data missing price fields")

                options_chain = defs_nearest.merge(latest_quotes[['premium']], left_on='symbol', right_index=True, how='left')
                options_chain = options_chain.dropna(subset=['premium'])
                return options_chain, S, T

            quotes_df = pd.concat(quotes_frames, ignore_index=True)
            latest_quotes = quotes_df.groupby('symbol').last()
            latest_quotes['premium'] = (latest_quotes['bid_px_00'] / 100.0 + latest_quotes['ask_px_00'] / 100.0) / 2

            options_chain = defs_nearest.merge(latest_quotes[['premium']], left_on='symbol', right_index=True, how='left')
            options_chain = options_chain.dropna(subset=['premium'])

            if relaxed_used:
                options_chain['matched_underlying_relaxed'] = True
            else:
                options_chain['matched_underlying_relaxed'] = False
            return options_chain, S, T

        except Exception as e:
            raise

    def _infer_option_type(self, row):
        """Infer option type (call/put) from instrument class or symbol"""
        instrument_class = row.get('instrument_class')
        symbol_value = row.get('symbol', '')

        if isinstance(instrument_class, (bytes, bytearray, np.bytes_)):
            try:
                ic_char = instrument_class.decode(errors='ignore').strip()[:1].upper()
                if ic_char == 'C':
                    return 'call'
                if ic_char == 'P':
                    return 'put'
            except Exception:
                pass
        elif isinstance(instrument_class, str):
            ic_char = instrument_class.strip()[:1].upper()
            if ic_char == 'C':
                return 'call'
            if ic_char == 'P':
                return 'put'

        if isinstance(symbol_value, (bytes, bytearray, np.bytes_)):
            try:
                symbol_value = symbol_value.decode(errors='ignore')
            except Exception:
                symbol_value = str(symbol_value)

        if isinstance(symbol_value, str):
            tokens = symbol_value.strip().split(' ')
            for token in tokens:
                if token and token[0].upper() in ('C', 'P'):
                    return 'call' if token[0].upper() == 'C' else 'put'

        return np.nan

    def black76_iv(self, premium: float, S: float, K: float, T: float, r: float, option_type: str):
        """Calculate implied volatility using Black-76 model"""
        def black76_price(sigma):
            if sigma <= 0 or T <= 0:
                return float('inf')

            d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
            d2 = d1 - sigma * np.sqrt(T)

            if option_type == 'call':
                theoretical_price = np.exp(-r * T) * (S * norm.cdf(d1) - K * norm.cdf(d2))
            else:
                theoretical_price = np.exp(-r * T) * (K * norm.cdf(-d2) - S * norm.cdf(-d1))

            return theoretical_price - premium

        try:
            return brentq(black76_price, 0.001, 5.0)
        except ValueError:
            return np.nan

    def calculate_rnsk(self, underlying: str, date: str = None, market_structure: str = None, risk_free_rate: float = 0.04):
        """Calculate Risk-Neutral Skewness using full density method"""
        if date is None:
            date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

        try:
            options_chain, S, T = self.fetch_options_data(underlying, date)

            # Calculate IVs (optional for density, but kept for variance fallback)
            try:
                options_chain['iv'] = options_chain.apply(
                    lambda row: self.black76_iv(row['premium'], S, row['strike'], T, risk_free_rate, row['option_type']),
                    axis=1
                )
            except Exception:
                pass
            options_chain_with_iv = options_chain.dropna(subset=['iv']) if 'iv' in options_chain.columns else pd.DataFrame()

            # OTM sets
            otm_calls_all = options_chain[(options_chain['option_type'] == 'call') & (options_chain['strike'] > S)]
            otm_puts_all = options_chain[(options_chain['option_type'] == 'put') & (options_chain['strike'] < S)]

            # Model-free variance fallback
            rnsk_var_based = np.nan
            V_E = np.nan
            V_L = np.nan
            if not options_chain_with_iv.empty:
                otm_calls_iv = options_chain_with_iv[(options_chain_with_iv['option_type'] == 'call') & (options_chain_with_iv['strike'] > S)]
                otm_puts_iv = options_chain_with_iv[(options_chain_with_iv['option_type'] == 'put') & (options_chain_with_iv['strike'] < S)]
                V_E = self._model_free_variance(otm_calls_iv, S, risk_free_rate, T)
                V_L = self._model_free_variance(otm_puts_iv, S, risk_free_rate, T)
                if not (np.isnan(V_E) or np.isnan(V_L) or V_L <= 0):
                    rnsk_var_based = 3 * (V_E - V_L) / (V_L ** 1.5)

            # Full Breeden-Litzenberger density-based skew
            rn_mean = np.nan
            rn_var = np.nan
            rn_skew = np.nan
            try:
                if otm_calls_all.empty or otm_puts_all.empty:
                    raise ValueError("Not enough OTM calls/puts")

                # Get unique sorted strikes
                strikes_unique = np.sort(options_chain['strike'].unique())
                if len(strikes_unique) < 5:  # Need more for both sides
                    raise ValueError("Insufficient strike grid")

                # Average strike spacing (handle uneven)
                dK_avg = np.mean(np.diff(strikes_unique))

                # PDF from puts (K < S)
                put_strikes = np.sort(otm_puts_all['strike'].unique())
                K_put_vals = []
                pdf_put_vals = []
                for i in range(1, len(put_strikes) - 1):
                    K = put_strikes[i]
                    dK_local = (put_strikes[i+1] - put_strikes[i-1]) / 2
                    P_minus = otm_puts_all[otm_puts_all['strike'] == put_strikes[i-1]]['premium'].values
                    P_mid = otm_puts_all[otm_puts_all['strike'] == K]['premium'].values
                    P_plus = otm_puts_all[otm_puts_all['strike'] == put_strikes[i+1]]['premium'].values
                    if len(P_minus) > 0 and len(P_mid) > 0 and len(P_plus) > 0:
                        second_deriv = (P_plus[0] - 2 * P_mid[0] + P_minus[0]) / (dK_local ** 2)
                        if second_deriv > 0:
                            K_put_vals.append(K)
                            pdf_put_vals.append(second_deriv)

                # PDF from calls (K > S)
                call_strikes = np.sort(otm_calls_all['strike'].unique())
                K_call_vals = []
                pdf_call_vals = []
                for i in range(1, len(call_strikes) - 1):
                    K = call_strikes[i]
                    dK_local = (call_strikes[i+1] - call_strikes[i-1]) / 2
                    C_minus = otm_calls_all[otm_calls_all['strike'] == call_strikes[i-1]]['premium'].values
                    C_mid = otm_calls_all[otm_calls_all['strike'] == K]['premium'].values
                    C_plus = otm_calls_all[otm_calls_all['strike'] == call_strikes[i+1]]['premium'].values
                    if len(C_minus) > 0 and len(C_mid) > 0 and len(C_plus) > 0:
                        second_deriv = (C_plus[0] - 2 * C_mid[0] + C_minus[0]) / (dK_local ** 2)
                        if second_deriv > 0:
                            K_call_vals.append(K)
                            pdf_call_vals.append(second_deriv)

                # Combine PDF points
                K_vals = np.concatenate((K_put_vals, K_call_vals))
                pdf_vals = np.concatenate((pdf_put_vals, pdf_call_vals))
                sort_idx = np.argsort(K_vals)
                K_vals = K_vals[sort_idx]
                pdf_vals = pdf_vals[sort_idx]

                # Scale by e^{rT}
                pdf_vals *= np.exp(risk_free_rate * T)

                if len(pdf_vals) >= 5:  # Minimum for reliable moments
                    # Normalize PDF
                    area = np.trapz(pdf_vals, K_vals)
                    if area > 0:
                        pdf_vals /= area

                        # Compute moments
                        rn_mean = np.trapz(K_vals * pdf_vals, K_vals)
                        rn_var = np.trapz((K_vals - rn_mean) ** 2 * pdf_vals, K_vals)
                        rn_std = np.sqrt(max(rn_var, 0))
                        if rn_std > 0:
                            rn_skew = np.trapz((K_vals - rn_mean) ** 3 * pdf_vals, K_vals) / (rn_std ** 3)
            except Exception:
                pass

            # Choose final rnsk: prefer full density skew, fallback to variance
            final_rnsk = rn_skew if not np.isnan(rn_skew) else rnsk_var_based

            # Determine prediction based on RNSK and market structure
            if np.isnan(final_rnsk):
                prediction = "INSUFFICIENT_DATA"
            elif market_structure == "BACKWARDATION":
                # Backwardation: negative spread (front > back)
                if final_rnsk <= 0:
                    # Low RNSK: spread becomes more negative (widening backwardation)
                    prediction = "BACKWARDATION_WIDENING"
                else:
                    # High RNSK: spread flattens toward zero or positive (diminishing backwardation)
                    prediction = "BACKWARDATION_DIMINISHING"
            else:
                # Contango or Mixed: positive spread (back > front) or mixed
                if final_rnsk > 0:
                    # High RNSK: spread becomes more positive (widening contango)
                    prediction = "CONTANGO_WIDENING"
                else:
                    # Low RNSK: spread flattens toward zero or negative (diminishing contango)
                    prediction = "CONTANGO_DIMINISHING"

            result = {
                'rnsk': final_rnsk,
                'prediction': prediction,
                'call_variance': V_E,
                'put_variance': V_L,
                'futures_price': S,
                'time_to_expiration': T,
                'num_options': len(options_chain),
                'num_otm_calls': len(otm_calls_all),
                'num_otm_puts': len(otm_puts_all),
                'rn_mean': rn_mean,
                'rn_var': rn_var,
                'rn_skew': rn_skew
            }

            return result

        except Exception as e:
            return {
                'rnsk': np.nan,
                'prediction': "ERROR",
                'error': str(e)
            }

    def _model_free_variance(self, options: pd.DataFrame, S: float, r: float, T: float):
        """Calculate model-free variance using trapezoidal integration"""
        if options.empty or len(options) < 2:
            return np.nan

        options = options.sort_values('strike')
        strikes = options['strike'].values
        premiums = options['premium'].values

        integrand = premiums / (strikes ** 2)
        integral = np.trapz(integrand, strikes)

        variance = (2 / T) * np.exp(r * T) * integral
        return variance

print("✅ RNSKCalculator class loaded with full density method")